# Reproduce MKGformer on Merged-MMKG (Kaggle Offline)

**LƯU Ý VỀ DUNG LƯỢNG KAGGLE (QUAN TRỌNG):**
- Disk limit của `/kaggle/working` chỉ có ~19.5GB.
- Thư mục `images` nặng tới 18GB, nếu dùng lệnh `cp` (copy) thì sẽ **tràn ổ đĩa ngay lập tức**.
- Notebook này sử dụng **Linux Symlink (`ln -s`)**: Tốn 0 MB ổ cứng `working`, đọc trực tiếp từ `/kaggle/input`!
- Bạn chỉ cần upload file zip `mkgformer-mmkg.zip` (hoặc thư mục `mkgformer-mmkg`) làm Kaggle Dataset.

In [ ]:
import os
import glob

# Tự động nhận diện thư mục dataset trong /kaggle/input
possible_dirs = [
    '/kaggle/input/mkgformer-mmkg',
    '/kaggle/input/mmkg-thesis-payload'
]
KAGGLE_INPUT_DIR = None
for d in possible_dirs:
    if os.path.exists(d):
        KAGGLE_INPUT_DIR = d
        break
if not KAGGLE_INPUT_DIR:
    for d in glob.glob('/kaggle/input/*'):
        if os.path.exists(os.path.join(d, 'images')) or os.path.exists(os.path.join(d, 'MKGformer')):
            KAGGLE_INPUT_DIR = d
            break

assert KAGGLE_INPUT_DIR, '❌ Không tìm thấy dataset trong /kaggle/input! Hãy kiểm tra xem bạn đã Add Dataset vào notebook chưa.'
print(f'✅ Đã tìm thấy Dataset tại: {KAGGLE_INPUT_DIR}')

WORKING_DIR = '/kaggle/working'

# 1. Chỉ copy DUY NHẤT source code MKGformer (~5MB) vì cần phẫu thuật code .py
!rm -rf {WORKING_DIR}/MKGformer
!cp -r {KAGGLE_INPUT_DIR}/MKGformer {WORKING_DIR}/

# 2. DÙNG SYMLINK (ln -s) CHO ẢNH, TRỌNG SỐ HF VÀ DATA -> TỐN 0 MB DISK QUOTA!
!mkdir -p {WORKING_DIR}/MKGformer/MKG/dataset
!ln -sfn {KAGGLE_INPUT_DIR}/images {WORKING_DIR}/MKGformer/MKG/dataset/FB15k-images
!ln -sfn {KAGGLE_INPUT_DIR}/MKGformer-Merged-MMKG {WORKING_DIR}/MKGformer/MKG/dataset/FB15k-237
!ln -sfn {KAGGLE_INPUT_DIR}/hf_weights {WORKING_DIR}/hf_weights

print('✅ Setup thành công! Dung lượng /kaggle/working chỉ tốn vài MB, còn nguyên ~19.5GB trống để lưu checkpoint!')


In [ ]:
print("Tiến hành Hack (Phẫu thuật) mã nguồn MKGformer để vượt mặt Kaggle Offline...")
mkg_dir = f"{WORKING_DIR}/MKGformer/MKG"
hf_bert = f"{WORKING_DIR}/hf_weights/bert-base-uncased"
hf_clip = f"{WORKING_DIR}/hf_weights/clip-vit-base-patch32"

for filepath in glob.glob(os.path.join(mkg_dir, "**", "*.py"), recursive=True):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Replace hardcoded strings (Cực kỳ nhức nhối trong code MKGformer)
    content = content.replace("'openai/clip-vit-base-patch32'", f"'{hf_clip}'")
    content = content.replace('"openai/clip-vit-base-patch32"', f"'{hf_clip}'")
    content = content.replace("'bert-base-uncased'", f"'{hf_bert}'")
    content = content.replace('"bert-base-uncased"', f"'{hf_bert}'")
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Hack Source Code thành công! MKGformer giờ đã là Offline Model.")

In [ ]:
# Môi trường Kaggle đã có sẵn Torch, Transformers, Lightning nên bỏ qua pip install offline
# !cd /kaggle/working/MKGformer && pip install -r requirements.txt -q

# Chỉnh sửa file sh để truyền đúng param BERT Offline
!sed -i 's|bert-base-uncased|/kaggle/working/hf_weights/bert-base-uncased|g' /kaggle/working/MKGformer/MKG/scripts/pretrain_fb15k-237-image.sh
!sed -i 's|bert-base-uncased|/kaggle/working/hf_weights/bert-base-uncased|g' /kaggle/working/MKGformer/MKG/scripts/fb15k-237-image.sh

print("✅ Mọi thứ đã sẵn sàng để Training!")

In [ ]:
%%writefile run_pretrain.sh
cd /kaggle/working/MKGformer/MKG
bash scripts/pretrain_fb15k-237-image.sh

In [ ]:
# PHASE 1: Entity Modeling (Train khoảng 1-2 tiếng)
!bash run_pretrain.sh

⚠️ **LƯU Ý QUAN TRỌNG:**
Sau khi Phase 1 chạy xong, nó sẽ lưu 1 cục tạ (checkpoint) vào thư mục `MKGformer/MKG/lightning_logs/version_0/checkpoints/`.
Anh mở thư mục đó ra, copy tên của file `.ckpt` rồi dán đè vào chữ `your_checkpoint_path` trong file `scripts/fb15k-237-image.sh`.
Sau đó mới được chạy Phase 2 bên dưới!

In [ ]:
%%writefile run_predict.sh
cd /kaggle/working/MKGformer/MKG
bash scripts/fb15k-237-image.sh

In [ ]:
# PHASE 2: Missing Entity Prediction (Link Prediction)
!bash run_predict.sh